# Fine-tuning Llama 3.2 3B on ATP 6-0.5 with Unsloth

This notebook demonstrates how to fine-tune a Small Language Model (SLM) on military manual data using Unsloth and QLoRA.

## What we'll cover:
1. Setup and installation
2. Data preparation
3. Model loading with Unsloth
4. Fine-tuning with QLoRA
5. Inference and evaluation
6. Edge deployment

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load and Prepare Data

In [ ]:
# Load training data
# If you don't have data yet, run: python src/prepare_data.py
dataset = load_dataset("json", data_files={
    "train": "../data/processed/train.jsonl",
    "test": "../data/processed/eval.jsonl"
})

print(f"Training examples: {len(dataset['train'])}")
print(f"Evaluation examples: {len(dataset['test'])}")
print("\nSample example:")
print(dataset['train'][0])

## 3. Load Model with Unsloth

Unsloth provides 2-5x faster training compared to standard methods!

In [ ]:
# Model configuration
max_seq_length = 2048
dtype = None  # Auto-detect (Float16 or BFloat16)
load_in_4bit = True  # Use 4bit quantization to reduce memory usage

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("✓ Model loaded successfully")

## 4. Apply LoRA Adapters

QLoRA uses 4-bit quantization + LoRA for efficient fine-tuning.

In [ ]:
# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
)

# Print trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

## 5. Prepare Data for Training

In [ ]:
# Define prompt template
prompt_template = """### Instruction:
You are an expert on ATP 6-0.5 (Mission Command). Answer the following question based on the manual.

### Question:
{input}

### Answer:
{output}"""

def formatting_func(examples):
    texts = []
    for i in range(len(examples["input"])):
        text = prompt_template.format(
            input=examples["input"][i],
            output=examples["output"][i]
        )
        texts.append(text)
    return {"text": texts}

# Apply formatting
train_dataset = dataset["train"].map(
    formatting_func,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Sample formatted text:")
print(train_dataset[0]["text"][:500] + "...")

## 6. Configure Training

In [ ]:
training_args = TrainingArguments(
    output_dir="../models/llama-3.2-3b-atp",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="tensorboard",
)

## 7. Start Training

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
)

# Start training
print("Starting training...")
trainer.train()

## 8. Save the Model

In [ ]:
# Save LoRA adapters
model.save_pretrained("../models/llama-3.2-3b-atp")
tokenizer.save_pretrained("../models/llama-3.2-3b-atp")

# Save merged model
model.save_pretrained_merged(
    "../models/llama-3.2-3b-atp-merged",
    tokenizer,
    save_method="merged_16bit"
)

print("✓ Model saved successfully")

## 9. Test Inference

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

# Test question
question = "What is mission command according to ATP 6-0.5?"

prompt = f"""### Instruction:
You are an expert on ATP 6-0.5 (Mission Command). Answer the following question based on the manual.

### Question:
{question}

### Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract answer
if "### Answer:" in response:
    answer = response.split("### Answer:")[-1].strip()
    print("Question:", question)
    print("\nAnswer:")
    print(answer)

## 10. Export for Edge Deployment

Convert to GGUF format for deployment on edge devices.

In [ ]:
# Export to GGUF (quantized format for edge deployment)
model.save_pretrained_gguf(
    "../models/llama-3.2-3b-atp-gguf",
    tokenizer,
    quantization_method="q4_k_m"  # 4-bit quantization
)

print("✓ Model exported to GGUF format for edge deployment")
print("\nYou can now deploy this model on edge devices using:")
print("- llama.cpp")
print("- Ollama")
print("- LM Studio")
print("- Or any other GGUF-compatible inference engine")

## Summary

In this notebook, we:
1. ✅ Loaded Llama 3.2 3B with Unsloth optimization
2. ✅ Applied QLoRA (4-bit quantization + LoRA) for efficient training
3. ✅ Fine-tuned on ATP 6-0.5 data
4. ✅ Saved the model in multiple formats
5. ✅ Tested inference
6. ✅ Exported for edge deployment

### Next Steps:
- Test on more diverse queries
- Fine-tune hyperparameters
- Deploy to edge hardware
- Create evaluation metrics
- Add more training data